In [1]:
import argparse
import os
from pathlib import Path
from omegaconf import OmegaConf
from itertools import chain
from toolz import identity, pipe
from toolz.curried import map as map_c

import numpy as np
import pandas as pd
from scipy import stats

import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, ConcatDataset
from lightning import LightningModule, Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
# from torchmetrics.regression import ...
from torchdiffeq import odeint # odeint_adjoint as odeint
from filterpy.kalman import UnscentedKalmanFilter, MerweScaledSigmaPoints

import plotly.graph_objects as go

from experiment.motion_sense.utils.dataset import TrajectoryDataset
from experiment.motion_sense.utils.field import FieldLitModule, Field
from experiment.motion_sense.utils.metric import SmoothedTrajectoryLoss

import mlflow

/home/semkin@ap-team.ru/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-ljlbf53b because there was an issue with the default path (/home/semkin@ap-team.ru/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
/home/semkin@ap-team.ru/n_ode/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SUBJ = [2, 0, 3, 4, 5, 6]

In [3]:
config = OmegaConf.load("/home/semkin@ap-team.ru/n_ode/experiment/motion_sense/config.yaml")

## Intra classification

In [4]:
intra_cls_df = []
for subj in SUBJ:
    cls_df = pd.read_csv(
        os.path.join(config.results_dir, str(subj), "cls.csv")
    )
    cls_df["subj"] = subj
    intra_cls_df.append(cls_df)
intra_cls_df = pd.concat(intra_cls_df, ignore_index=True)
intra_cls_df.head()

,act_true,act,loss,subj
0,ups,ups,0.689101,2
1,ups,std,1.137364,2
2,ups,sit,0.943902,2
3,ups,wlk,1.048421,2
4,ups,dws,0.954913,2


In [5]:
argmin_indx = intra_cls_df.groupby(["act_true", "subj"])["loss"].idxmin()
intra_cls_df = intra_cls_df.loc[argmin_indx]
intra_cls_df["is_correct"] = intra_cls_df["act_true"] == intra_cls_df["act"]
intra_cls_df.head()

,act_true,act,loss,subj,is_correct
60,dws,ups,1.724247,0,False
28,dws,dws,0.724642,2,True
98,dws,sit,2.245008,3,False
134,dws,sit,1.149982,4,False
168,dws,ups,1.313025,5,False


In [13]:
intra_cls_df

,act_true,act,loss,subj,is_correct
60,dws,ups,1.724247,0,False
28,dws,dws,0.724642,2,True
98,dws,sit,2.245008,3,False
134,dws,sit,1.149982,4,False
168,dws,ups,1.313025,5,False
206,dws,sit,1.131869,6,False
70,jog,dws,3.911207,0,False
35,jog,jog,1.742714,2,True
107,jog,jog,3.314700,3,True
143,jog,jog,2.246025,4,True


In [6]:
intra_cls_df.groupby("act_true")["is_correct"].mean()

act_true
dws    0.166667
jog    0.833333
sit    1.000000
std    0.666667
ups    0.666667
wlk    0.833333
Name: is_correct, dtype: float64

In [7]:
intra_cls_df["is_correct"].mean()

np.float64(0.6944444444444444)

## Inter classification

In [8]:
inter_cls_df = []
for act in config.activity_codes:
    cls_df = pd.read_csv(
        os.path.join(config.results_dir, f"{act}_inter.csv")
    )
    cls_df["act"] = act
    inter_cls_df.append(cls_df)
inter_cls_df = pd.concat(inter_cls_df, ignore_index=True)
inter_cls_df.head()

,subj_true,subj,loss,act
0,4,4,0.700962,dws
1,4,6,0.890007,dws
2,4,2,0.824967,dws
3,4,0,1.060267,dws
4,4,5,1.125629,dws


In [9]:
argmin_indx = inter_cls_df.groupby(["subj_true", "act"])["loss"].idxmin()
inter_cls_df = inter_cls_df.loc[argmin_indx]
inter_cls_df["is_correct"] = inter_cls_df["subj_true"] == inter_cls_df["subj"]
inter_cls_df.head()

,subj_true,subj,loss,act,is_correct
18,0,4,1.046243,dws,False
59,0,3,2.433231,jog,False
198,0,4,0.029074,sit,False
164,0,2,0.030115,std,False
94,0,5,0.891808,ups,False


In [10]:
inter_cls_df.groupby("act")["is_correct"].mean()

act
dws    0.666667
jog    0.833333
sit    0.333333
std    0.666667
ups    0.666667
wlk    1.000000
Name: is_correct, dtype: float64

In [11]:
inter_cls_df["is_correct"].mean()

np.float64(0.6944444444444444)

In [12]:
for act, group in inter_cls_df.groupby("act"):
    test_res = stats.binomtest(
        group["is_correct"].sum(), group.shape[0], p=0.5, alternative="greater"
    )
    print(act, test_res.pvalue)

dws 0.34375
jog 0.109375
sit 0.890625
std 0.34375
ups 0.34375
wlk 0.015625


In [14]:
test_res = stats.binomtest(
    inter_cls_df["is_correct"].sum(), inter_cls_df.shape[0], p=0.5, alternative="greater"
)
print(act, test_res.pvalue)

wlk 0.014408359827939423
